# 第 7 周 · 笔记本 3：用 QLoRA 微调（Fine-tuning）

## 练习目标

用 **QLoRA**（4-bit 量化 + LoRA 适配器）微调 **Llama 3.2**，做价格预测任务：

1. 以 4-bit 加载基础模型（Base Model）
2. 配置 LoRA 适配器（只训练少量参数）
3. 用 `SFTTrainer`（Supervised Fine-Tuning）训练
4. 把适配器保存到本地，并推到 Hugging Face Hub

**预期结果：** 约 $40 误差量级（课程叙事：挑战 GPT-5.1）

## 预计耗时：2–12 小时（取决于 LITE/FULL 与 GPU）

**重要：** 请在 **Google Colab + GPU** 上跑！

- 免费 T4：约 3 小时（精简 / LITE 模式）
- 付费 A100：约 8–12 小时（完整 / FULL 模式）

## Google Colab 设置

1. 把本笔记本上传到 Google Colab
2. 运行时 → 更改运行时类型 → GPU（T4 或 A100）
3. 从上到下运行所有单元格（记得把 `HF_TOKEN` / `HUB_MODEL_ID` 换成你的）


## 安装依赖（仅 Colab 需要）


In [ ]:
# ========== 可选安装：在 Google Colab 上取消注释后运行 ==========
# torch / transformers / datasets / peft / trl / bitsandbytes / accelerate 等是 QLoRA 训练栈
# Uncomment and run on Google Colab
# !pip install -q torch transformers datasets peft trl bitsandbytes accelerate python-dotenv huggingface-hub pydantic


## 设置环境（登录 Hub + 检查 GPU）


In [ ]:
# ========== 导入训练栈 + Hugging Face 登录 + GPU 自检 ==========

# os：环境变量等（本格主要为后续扩展预留）
import os
# torch：检测 CUDA、读 GPU 名称与显存
import torch
# login：用 token 登录 Hugging Face Hub（拉 gated 模型 / 推送适配器）
from huggingface_hub import login
# AutoTokenizer / AutoModelForCausalLM：按 model id 加载分词器与因果语言模型
# BitsAndBytesConfig：4-bit 量化配置；TrainingArguments：Trainer 超参
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
# LoRA：低秩适配；prepare_model_for_kbit_training：为 k-bit 训练做准备
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# SFTTrainer：监督微调封装（TRL）
from trl import SFTTrainer
# load_dataset：从 Hub 拉训练/验证集
from datasets import load_dataset

# ---------- 运行开关与密钥 ----------
# LITE_MODE=True：小数据/少 epoch，方便先跑通；改 False 做完整训练
LITE_MODE = True  # Set to False for full training
# 换成你的 Hugging Face token（字符串占位符保持原样，勿提交真密钥到公开仓库）
HF_TOKEN = "your_token_here"  # Replace with your HuggingFace token

# 登录 Hub，后续才能拉 Llama 等需授权的模型
login(HF_TOKEN)

print("✅ Environment setup complete")
# 是否看得到 CUDA GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    # 打印第一块 GPU 名称与总显存（GB）
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 配置：模型 / 数据集 / LoRA / 训练超参


In [ ]:
# ========== 集中配置：LITE vs FULL 两套档位 ==========

# ---------- 模型与数据路径 ----------
# 基础因果语言模型 id（需有权访问 meta-llama）
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# LITE 用小数据集，FULL 用完整集（Hub 数据集名不可改译）
DATASET_NAME = "ed-donner/items_prompts_lite" if LITE_MODE else "ed-donner/items_prompts_full"
# 本地保存目录
OUTPUT_DIR = "./llama-pricer-lite" if LITE_MODE else "./llama-pricer-full"
# 推送到 Hub 的仓库 id：请把 your-username 换成你的用户名
HUB_MODEL_ID = "your-username/llama-pricer-lite" if LITE_MODE else "your-username/llama-pricer-full"

# ---------- QLoRA / LoRA 超参 ----------
# r：低秩维度；越大表达力越强、可训练参数也越多
LORA_RANK = 16 if LITE_MODE else 32
# alpha：LoRA 缩放相关；常取 2*r
LORA_ALPHA = 32 if LITE_MODE else 64
# dropout：LoRA 层丢弃率，减轻过拟合
LORA_DROPOUT = 0.05

# ---------- 训练超参 ----------
BATCH_SIZE = 4 if LITE_MODE else 8
# 梯度累积：有效 batch ≈ BATCH_SIZE * GRADIENT_ACCUMULATION
GRADIENT_ACCUMULATION = 4
NUM_EPOCHS = 3 if LITE_MODE else 5
LEARNING_RATE = 2e-4
# 单条序列最大长度（tokenize 后截断/填充到此）
MAX_SEQ_LENGTH = 256

# 打印当前档位摘要，方便开跑前核对
print("Configuration:")
print(f"  Mode: {'LITE' if LITE_MODE else 'FULL'}")
print(f"  Dataset: {DATASET_NAME}")
print(f"  LoRA Rank: {LORA_RANK}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")


## 加载数据集


In [ ]:
# ========== 从 Hub 加载 prompt/completion 数据集 ==========

print(f"Loading dataset: {DATASET_NAME}")
# 按 DATASET_NAME 拉取；通常含 train / val 等 split
dataset = load_dataset(DATASET_NAME)

# 取出训练与验证划分
train_data = dataset["train"]
val_data = dataset["val"]

print(f"✅ Dataset loaded:")
print(f"   Training: {len(train_data):,} examples")
print(f"   Validation: {len(val_data):,} examples")

# ---------- 偷看第一条，确认字段是 prompt + completion ----------
print(f"\nExample training data:")
# 只打印 prompt 前 100 字符，避免刷屏
print(f"Prompt: {train_data[0]['prompt'][:100]}...")
print(f"Completion: {train_data[0]['completion']}")


## 用 4-bit 量化加载基础模型（QLoRA 的 Q）


In [ ]:
# ========== BitsAndBytes 4-bit + 分词器 + 因果 LM ==========

# nf4 + double quant：常见 QLoRA 量化配方；计算用 float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {BASE_MODEL}")
print("This may take a few minutes...")

# 加载与模型匹配的分词器
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
# 若没有 pad_token，借用 eos_token，避免批训练报错
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 按 4-bit 配置加载因果语言模型；device_map="auto" 自动切设备
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 模型配置与分词器 pad id 对齐
model.config.pad_token_id = tokenizer.pad_token_id
# 梯度检查点需要关闭 use_cache（KV cache 与反传不兼容）
model.config.use_cache = False  # Required for gradient checkpointing

print("✅ Model loaded in 4-bit")
# 粗估显存占用（字节 → GB）
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")


## 配置 LoRA 适配器（只训练低秩旁路）


In [ ]:
# ========== 为 k-bit 训练准备模型，再挂上 LoRA ==========

# 冻结量化底座、开启必要梯度等（PEFT 推荐步骤）
model = prepare_model_for_kbit_training(model)

# r / alpha / dropout 来自配置格；target_modules 对准注意力投影层
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

# 把 LoRA 适配器注入基础模型 → PeftModel
model = get_peft_model(model, lora_config)

# 统计「可训练参数 / 总参数」，体会 LoRA 的参数效率
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable_params / total_params

print("✅ LoRA adapters configured")
print(f"Trainable parameters: {trainable_params:,} ({trainable_pct:.2f}%)")
print(f"Total parameters: {total_params:,}")


## 配置训练参数（TrainingArguments）


In [ ]:
# ========== Hugging Face TrainingArguments：优化器 / 日志 / 保存策略 ==========

training_args = TrainingArguments(
    # 检查点与最终权重输出目录
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    # 累积多步再更新，放大有效 batch
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    # 余弦学习率调度
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=10,
    save_steps=500,
    eval_steps=500,
    # 按 steps 做评估与保存；结束时加载最佳
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    # 混合精度 + 梯度检查点：省显存
    fp16=True,
    gradient_checkpointing=True,
    # 8-bit paged AdamW：QLoRA 常见优化器选择
    optim="paged_adamw_8bit",
    report_to="none",
    # 先不自动 push；后面手动 push_to_hub
    push_to_hub=False,  # We'll push manually later
)

print("✅ Training arguments configured")


## 设置训练样本格式（prompt + completion 拼接）


In [ ]:
# ========== formatting_func：SFTTrainer 用来把一条样本变成训练文本 ==========

def formatting_func(example):
    """Format prompt-completion pairs for training"""
    # 简单拼接：问题/上下文 + 目标补全（字符串字段名不可改）
    return example["prompt"] + example["completion"]

print("✅ Formatting function ready")


## 创建 SFTTrainer


In [ ]:
# ========== 组装 SFTTrainer：模型 + 数据 + 分词器 + 格式化函数 ==========

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    formatting_func=formatting_func,
    # 与配置格 MAX_SEQ_LENGTH 一致
    max_seq_length=MAX_SEQ_LENGTH,
)

print("✅ Trainer created")
print(f"\nReady to train for {NUM_EPOCHS} epochs on {len(train_data):,} examples")
# 预估耗时文案保持原样（依赖 LITE_MODE）
print(f"Estimated time: {'2-3 hours' if LITE_MODE else '8-12 hours'}")


## 开始训练！


In [ ]:
# ========== 真正开训：阻塞直到 epoch 跑完 ==========

print("Starting training...")
print("This will take a while. Go get coffee! ☕")
print("="*60)

# trainer.train()：按 TrainingArguments 循环 forward / backward / eval / save
trainer.train()

print("="*60)
print("✅ Training complete!")


## 保存模型到本地


In [ ]:
# ========== 本地落盘：适配器权重 + 分词器文件 ==========

print(f"Saving model to {OUTPUT_DIR}...")
# 保存 Peft/训练后的模型文件到 OUTPUT_DIR
trainer.save_model(OUTPUT_DIR)
# 分词器一并保存，推理时需配套
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Model saved locally")


## 推送到 Hugging Face Hub


In [ ]:
# ========== 上传到 Hub：方便别处 from_pretrained 拉取 ==========

print(f"Pushing to HuggingFace Hub: {HUB_MODEL_ID}")
# use_auth_token=True：使用当前 login 会话（参数名保持原样）
model.push_to_hub(HUB_MODEL_ID, use_auth_token=True)
tokenizer.push_to_hub(HUB_MODEL_ID, use_auth_token=True)

print("✅ Model pushed to Hub")
# URL 模板依赖 HUB_MODEL_ID，不可改掉域名结构
print(f"\n🔗 View at: https://huggingface.co/{HUB_MODEL_ID}")


## 小结

✅ 微调流程跑完！

**我们做了什么：**

1. 以 **4-bit 量化**加载 Llama 3.2 底座
2. 挂上 **LoRA** 适配器（只训练很小比例的参数）
3. 用 **SFTTrainer** 在 prompt–completion 对上监督微调
4. 把适配器保存到本地，并推到 **Hugging Face Hub**

**预期结果（课程叙事）：**

- Lite 模式：约 ~$65 误差
- Full 模式：约 ~$40 误差（叙事上挑战 GPT-5.1）

**下一步：** 打开 `04_evaluation.ipynb`，评估微调后的模型！
